# PySpark on Kubernetes - Scalable Cluster Example

This notebook demonstrates how to connect PySpark to the **Kubernetes Control Plane**, allowing Spark to dynamically spawn and autoscale executor pods on demand.

In [ ]:
import os
from pyspark.sql import SparkSession

# Create SparkSession connected to Kubernetes
spark = SparkSession.builder \
    .appName("k8s-pyspark-scaling-demo") \
    .master("k8s://https://kubernetes.default.svc:443") \
    .config("spark.kubernetes.container.image", "pyspark-notebook-k8s:latest") \
    .config("spark.kubernetes.container.image.pullPolicy", "IfNotPresent") \
    .config("spark.kubernetes.namespace", "spark") \
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark-driver") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "1g") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "5") \
    .getOrCreate()

print("Spark Context initialized successfully!")
print("Spark Master:", spark.sparkContext.master)
print("Spark App ID:", spark.sparkContext.applicationId)

## Distributed Computation Test: Monte Carlo Estimation of Pi

The following code distributes 100,000,000 samples across the Kubernetes Spark executor pods to estimate Pi in parallel.

In [ ]:
import random
import time

NUM_SAMPLES = 100000000
NUM_SLICES = 100

def sample(p):
    x = random.random()
    y = random.random()
    return 1 if x*x + y*y < 1 else 0

start_time = time.time()
count = spark.sparkContext.parallelize(range(0, NUM_SAMPLES), NUM_SLICES).map(sample).reduce(lambda a, b: a + b)
elapsed = time.time() - start_time

pi_estimate = 4.0 * count / NUM_SAMPLES
print(f"Pi Estimate: {pi_estimate}")
print(f"Computation completed in {elapsed:.2f} seconds across executor pods!")

## Distributed DataFrame Operations & Repartitioning

In [ ]:
from pyspark.sql import functions as F

# Generate 10 Million Rows of Synthetic Data
df = spark.range(0, 10000000).repartition(20)
df = df.withColumn("category", (F.col("id") % 10).cast("string")) \
       .withColumn("value", F.rand() * 100)

print("Total partition count:", df.rdd.getNumPartitions())

# Perform GroupBy Aggregation across Executors
summary = df.groupBy("category").agg(
    F.count("*").alias("record_count"),
    F.avg("value").alias("avg_value"),
    F.max("value").alias("max_value")
).orderBy("category")

summary.show()

In [ ]:
# Stop Spark Context when finished
# spark.stop()